In [1]:
import pandas as pd
import joblib


In [3]:
df_a = pd.read_csv('C:/Users/Rashmi S/Desktop/crop-recommendation-system/data/raw/Crop_recommendation.csv')
df_b = pd.read_csv('C:/Users/Rashmi S/Desktop/crop-recommendation-system/data/raw/crop_production_india.csv')

In [6]:
# --- Standardize crop names in B ---
df_b['Crop'] = df_b['Crop'].str.lower().str.strip()
df_b['Season'] = df_b['Season'].str.strip()

In [7]:
# --- Mapping B crop names → A crop names ---
crop_name_mapping = {
    'rice'             : 'rice',
    'maize'            : 'maize',
    'gram'             : 'chickpea',
    'arhar/tur'        : 'pigeonpeas',
    'moong(green gram)': 'mungbean',
    'urad'             : 'blackgram',
    'lentil'           : 'lentil',
    'banana'           : 'banana',
    'mango'            : 'mango',
    'grapes'           : 'grapes',
    'watermelon'       : 'watermelon',
    'muskmelon'        : 'muskmelon',
    'apple'            : 'apple',
    'orange'           : 'orange',
    'papaya'           : 'papaya',
    'coconut'          : 'coconut',
    'cotton(lint)'     : 'cotton',
    'jute'             : 'jute',
    'coffee'           : 'coffee',
    'kidney beans'     : 'kidneybeans',
    'moth beans'       : 'mothbeans',
    'pomegranate'      : 'pomegranate',
    'wheat'            : 'wheat',
}

df_b['label'] = df_b['Crop'].map(crop_name_mapping)

In [8]:
# --- Get most common season per crop from B ---
most_common_season = (
    df_b.dropna(subset=['label'])
    .groupby('label')['Season']
    .agg(lambda x: x.value_counts().index[0])
    .reset_index()
    .rename(columns={'Season': 'season'})
)

print("\nSeason lookup table:")
print(most_common_season)


Season lookup table:
         label      season
0        apple  Whole Year
1       banana  Whole Year
2    blackgram      Kharif
3     chickpea        Rabi
4      coconut  Whole Year
5       coffee  Whole Year
6       cotton      Kharif
7       grapes  Whole Year
8         jute      Kharif
9       lentil        Rabi
10       maize      Kharif
11       mango  Whole Year
12    mungbean      Kharif
13      orange  Whole Year
14      papaya  Whole Year
15  pigeonpeas      Kharif
16        rice      Kharif
17       wheat        Rabi


In [9]:
# --- Merge season into A ---
df_merged = df_a.merge(most_common_season, 
                        on='label', how='left')

In [10]:
# --- Fill any crops that didn't match B manually ---
manual_seasons = {
    'kidneybeans' : 'Rabi',
    'mothbeans'   : 'Kharif',
    'mungbean'    : 'Kharif',
    'pigeonpeas'  : 'Kharif',
    'pomegranate' : 'Whole Year',
    'grapes'      : 'Whole Year',
    'muskmelon'   : 'Summer',
    'watermelon'  : 'Summer',
    'chickpea'    : 'Rabi',
    'lentil'      : 'Rabi',
    'blackgram'   : 'Kharif',
    'coffee'      : 'Whole Year',
    'coconut'     : 'Whole Year',
    'papaya'      : 'Whole Year',
    'banana'      : 'Whole Year',
}

for crop, season in manual_seasons.items():
    mask = (df_merged['label'] == crop) & (df_merged['season'].isnull())
    df_merged.loc[mask, 'season'] = season

In [11]:
print("\nNull seasons remaining:", df_merged['season'].isnull().sum())
print("\nSeason distribution:")
print(df_merged['season'].value_counts())
print("\nSample rows:")
print(df_merged[['label', 'season']].drop_duplicates())


Null seasons remaining: 0

Season distribution:
season
Whole Year    900
Kharif        800
Rabi          300
Summer        200
Name: count, dtype: int64

Sample rows:
            label      season
0            rice      Kharif
100         maize      Kharif
200      chickpea        Rabi
300   kidneybeans        Rabi
400    pigeonpeas      Kharif
500     mothbeans      Kharif
600      mungbean      Kharif
700     blackgram      Kharif
800        lentil        Rabi
900   pomegranate  Whole Year
1000       banana  Whole Year
1100        mango  Whole Year
1200       grapes  Whole Year
1300   watermelon      Summer
1400    muskmelon      Summer
1500        apple  Whole Year
1600       orange  Whole Year
1700       papaya  Whole Year
1800      coconut  Whole Year
1900       cotton      Kharif
2000         jute      Kharif
2100       coffee  Whole Year


In [12]:
df_merged.to_csv('../data/processed/crop_india_merged.csv', index=False)
print("\n✅ Saved → data/processed/crop_india_merged.csv")
print("Final columns:", df_merged.columns.tolist())


✅ Saved → data/processed/crop_india_merged.csv
Final columns: ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall', 'label', 'season']
